E-24: controlled nonlinear benchmark (H24-Base, H24-GRR; replaces the old `tab:simulation2`)

Design: `doc/2026-10-07_experiment_registration.md` (parent repository), E-24, §1.2, §1.4, §1.5, §1.9 (family 7). U28: the design is not a binding registration; the choices it leaves open are in `grrexp/e24.py` and in `docs/spec/02` §2.1.

- `python3 notebooks/experiments/run_registered.py E-24 stage0 22_E24_controlled_nonlinear_benchmark.ipynb`
- `... E-24 stage0.5 ...`, then `... E-24 stage1 ...` (each after the previous stage is committed).

Stage 0 checks the design constants by quadrature and freezes the tuning on the pilot data of each $n$ (stream 5): every later stage loads it. A solver that raises stops the notebook (§1.1).

In [1]:
import io
import json
import os
import time

import pandas as pd

from grrexp import env, outputs

gr = env.use_submodule_genriesz()
from grrexp import baselines, e24, parallel  # noqa: E402
from grrexp.seeds import CONFIRMATORY_ENTROPY, PILOT_ENTROPY, Seeds  # noqa: E402

STAGE = json.loads(os.environ[outputs.RUN_ENV])["stage"]
rec = outputs.RunRecorder(24, STAGE, cells=None if STAGE == "stage0" else e24.CELLS,
                          arms=None if STAGE == "stage0" else [a.replace("|", "-") for a in e24.ARM_LABELS])
STAGE

'stage0'

# Stage 0: design checks and the frozen tuning (pilot data of each $n$, stream 5, 5-fold CV)

In [2]:
if STAGE == "stage0":
    design = e24.stage0()
    print(json.dumps({k: design[k] for k in ("checks", "ok", "differences")}, indent=1))
    if not design["ok"]:
        raise RuntimeError("DGP24 does not reproduce the registered design constants")
    parallel.configure_worker()
    t0 = time.perf_counter()
    tunings = parallel.run_tasks(e24.tune_task, [(c, CONFIRMATORY_ENTROPY) for c in range(len(e24.CELLS))],
                                 len(e24.CELLS))
    tuning_seconds = time.perf_counter() - t0
    rec.write_stage0("e24_design", {"design": design, "tuning": [json.loads(t) for t in tunings],
                                    "tuning_seconds": tuning_seconds})
    for t in tunings:
        t = json.loads(t)
        print(t["n"], t["lambda"], t["weight_decay"], t["logit_C"], t["outcome_lambda"], t["gbm"],
              "CV failures", t["cv_failures"], t["warnings"])
    n_warn = sum(sum(json.loads(t)["warnings"].values()) for t in tunings)
    # failed CV fits of the tuning (each candidate value, fold and status is in the JSON)
    n_fail = sum(json.loads(t)["cv_failures"] for t in tunings)
    rec.finalize(R=None, n=None, warnings=int(n_warn), failures=int(n_fail))

{
 "checks": {
  "eta_bound_outward": true,
  "e_range_outward": true,
  "alpha0_bound_outward": true,
  "theta0": true,
  "quadrature_agreement": true
 },
 "ok": true,
 "differences": {
  "E_alpha0_sq": 1.5987211554602254e-14,
  "theta0": 4.440892098500626e-16,
  "var_tau": 3.885780586188048e-16,
  "V_star": 1.687538997430238e-14
 }
}


1000 {'BKL1': 0.1, 'BP_C0': 0.001, 'SQ': 0.001, 'UKL1': 0.01} 0.01 1.0 0.0001 {'outcome': [0.05, 7, 100], 'propensity': [0.05, 7, 100], 'seed': 1917009632} CV failures 15 {}
2000 {'BKL1': 0.01, 'BP_C0': 0.001, 'SQ': 0.0001, 'UKL1': 0.001} 0.01 1.0 0.0001 {'outcome': [0.1, 7, 100], 'propensity': [0.05, 7, 100], 'seed': 1948982463} CV failures 10 {}
4000 {'BKL1': 0.1, 'BP_C0': 0.0001, 'SQ': 0.0001, 'UKL1': 0.001} 0.01 1.0 0.0001 {'outcome': [0.05, 7, 300], 'propensity': [0.05, 7, 100], 'seed': 2005592289} CV failures 15 {}
8000 {'BKL1': 0.1, 'BP_C0': 0.0001, 'SQ': 0.0001, 'UKL1': 0.0001} 0.01 100.0 0.0001 {'outcome': [0.05, 7, 300], 'propensity': [0.05, 15, 100], 'seed': 369500794} CV failures 12 {}


# The workflow after the replications (shared by Stage 0.5 and Stage 1)

The §1.5 metrics per cell, arm and estimator, and the comparisons of the ARW RMSE (family 7, stream 4, $B = 200000$, multinomial counts): H24-Base for each $n$, GRR arm, baseline; then H24-GRR for each $n$ and generator pair; a comparison with a common-success rate below 95% is not evaluable, consumes no draws and stays in its Holm family with $p = 1$.

In [3]:
def load_tunings():
    payload = json.loads((outputs.stage_dir(24, "stage0") / "stage0_e24_design.json").read_text())
    return [json.dumps(t, sort_keys=True) for t in payload["tuning"]]


def aggregate(raw, seeds):
    summ = e24.summarise(raw)
    comps = e24.compare(raw, seeds)
    summ["comparisons"] = json.dumps(comps)
    summ["families"] = json.dumps(e24.family_sentences(comps))
    return summ

# Stage 0.5: timing of the replications and the aggregation, status counts, 1-versus-12-worker identity (no estimates kept)

In [4]:
if STAGE == "stage0.5":
    import matplotlib.pyplot as plt

    parallel.configure_worker()
    tunings = load_tunings()
    timing, serial, para, all_tasks = [], [], [], []
    for workers in (1, outputs.PILOT_WORKERS):
        for ci, cell in enumerate(e24.CELLS):
            t0 = time.perf_counter()
            tasks = e24.tasks_for(PILOT_ENTROPY, outputs.PILOT_REPS, tunings, cells=[ci])
            res = parallel.run_tasks(e24.replicate, tasks, workers)
            aggregate(e24.raw_frame(tasks, res), Seeds(24, entropy=PILOT_ENTROPY))
            timing.append({"cell": cell, "workers": workers, "seconds": time.perf_counter() - t0})
            if workers == 1:
                serial += res
                all_tasks += tasks
            else:
                para += res
    digest = parallel.assert_bit_identical(serial, para, expected=len(all_tasks))
    raw = e24.raw_frame(all_tasks, serial)
    t0 = time.perf_counter()
    buf = io.StringIO()
    aggregate(raw, Seeds(24, entropy=PILOT_ENTROPY)).to_csv(buf, index=False)
    buf.seek(0)
    S = pd.read_csv(buf)
    e24.tables(S, json.loads(S["comparisons"].iloc[0]))
    plt.close(e24.figure(S))
    whole_seconds = time.perf_counter() - t0
    counts = raw.groupby(["cell", "arm", "status"]).size().reset_index(name="count")
    status_df = pd.DataFrame({"cell": [e24.CELLS[c] for c in counts["cell"]],
                              "arm": counts["arm"].str.replace("|", "-", regex=False),
                              "status": counts["status"], "count": counts["count"].astype(int)})
    rec.write_pilot("timing.csv", pd.DataFrame(timing))
    rec.write_pilot("status_counts.csv", status_df)
    rec.record_parallel_identity(cells=e24.CELLS, reps=outputs.PILOT_REPS, n_jobs=outputs.PILOT_WORKERS,
                                 digest=digest)
    print(pd.DataFrame(timing).groupby("workers")["seconds"].sum(), whole_seconds)
    print(status_df.groupby(["arm", "status"])["count"].sum().to_string())
    rec.finalize(R={"main": outputs.PILOT_REPS}, n={"main": list(e24.N_VALUES)},
                 warnings=e24.total_warnings(raw), failures=int((raw["status"] != "ok").sum()),
                 extra={"whole_pilot_aggregation_table_and_figure_seconds": whole_seconds})
    del raw

# Stage 1: confirmatory Monte Carlo

In [5]:
if STAGE == "stage1":
    parallel.configure_worker()
    tunings = load_tunings()
    tasks = e24.tasks_for(CONFIRMATORY_ENTROPY, e24.R, tunings)
    t0 = time.perf_counter()
    results = parallel.run_tasks(e24.replicate, tasks, outputs.PILOT_WORKERS)
    print("seconds", time.perf_counter() - t0)
    raw = e24.raw_frame(tasks, results)
    rec.write_raw(raw)
    n_warn = e24.total_warnings(raw)
    raw.shape, n_warn

In [6]:
if STAGE == "stage1":
    summ = aggregate(raw, Seeds(24))
    rec.write_summary(summ)
    json.loads(summ["families"].iloc[0])

# Tables, figure and macros (from summary.csv and the frozen tuning only, §4)

In [7]:
if STAGE == "stage1":
    S = outputs.read_summary(rec)
    comps = json.loads(S["comparisons"].iloc[0])
    for name, tex in e24.tables(S, comps).items():
        rec.write_table(name, tex)
    rec.write_table("tab_E24_tuning", e24.tuning_table(tunings))
    rec.save_figure(e24.figure(S), "fig_E24_rmse")
    rec.write_macros(e24.macros(S, comps))
    print(json.dumps(e24.family_sentences(comps), indent=1))
    rec.finalize(R={"main": e24.R}, n={"main": list(e24.N_VALUES)}, warnings=int(n_warn),
                 failures=int((S["R"] - S["R_s"]).sum()))